# Module 2: Data Cleaning and Structuring

## Tasks:
- Clean text: remove commas, %, +, special symbols
- Convert metrics to numeric formats
- Standardize column names
- Handle missing/null values
- Save a clean dataset for Tableau input

In [1]:
import pandas as pd
import numpy as np
import re
import os

### 1. Load the Raw Data

In [2]:
# Determine input path based on standard repo structure
input_path = '../Module 1/military_raw_data.csv'
if not os.path.exists(input_path):
    # Fallback to local if running differently
    input_path = 'military_raw_data.csv'

df = pd.read_csv(input_path)
df.head()

,Country,Power Index,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,...,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km
0,United States,0.0741,"341,963,408","150,463,900","124,816,644","4,445,524","1,333,030","799,500",0,"13,032",...,"1,029,000,000,000 \t\t\...","914,301,000,000 \t\t\t\...","13,402,000,000,000 \t\t...","534,234,000 \t\t\t\t\t\...","495,156,000 \t\t\t\t\t\...","247,883,000,000 \t\t\t\...","9,833,517 \t\t\t\t\t\t\...","19,924 \t\t\t\t\t\t\r\n...","12,002 \t\t\t\t\t\t\r\n...","41,009 \t\t\t\t\t\t\r\n..."
1,Russia,0.0791,"140,820,810","69,002,197","46,189,226","1,267,387","1,320,000","2,000,000","250,000","4,237",...,"617,830,000,000 \t\t\t\...","472,239,000,000 \t\t\t\...","47,805,000,000,000 \t\t...","531,130,000 \t\t\t\t\t\...","290,763,000 \t\t\t\t\t\...","162,166,000,000 \t\t\t\...","17,098,242 \t\t\t\t\t\t...","37,653 \t\t\t\t\t\t\r\n...","22,407 \t\t\t\t\t\t\r\n...","102,000 \t\t\t\t\t\t\r\..."
2,China,0.0919,"1,415,043,270","764,123,366","626,864,169","19,810,606","2,035,000","510,000","625,000","3,529",...,"225,341,000,000 \t\t\t\...","366,160,000,000 \t\t\t\...","6,654,000,000,000 \t\t\...","4,805,000,000 \t\t\t\t\...","5,191,000,000 \t\t\t\t\...","157,041,000,000 \t\t\t\...","9,596,960 \t\t\t\t\t\t\...","14,500 \t\t\t\t\t\t\r\n...","22,457 \t\t\t\t\t\t\r\n...","27,700 \t\t\t\t\t\t\r\n..."
3,India,0.1346,"1,409,128,296","662,290,299","522,786,598","23,955,181","1,431,000","1,000,000","2,527,000","2,183",...,"33,170,000,000 \t\t\t\t...","58,867,000,000 \t\t\t\t...","1,381,000,000,000 \t\t\...","1,020,000,000 \t\t\t\t\...","1,262,000,000 \t\t\t\t\...","127,727,000,000 \t\t\t\...","3,287,263 \t\t\t\t\t\t\...","7,000 \t\t\t\t\t\t\r\n\...","13,888 \t\t\t\t\t\t\r\n...","14,500 \t\t\t\t\t\t\r\n..."
4,South Korea,0.1642,"52,081,799","26,040,900","21,353,538","416,654","450,000","3,100,000","120,000","1,540",...,"55,127,000 \t\t\t\t\t\t...","59,480,000,000 \t\t\t\t...","7,079,000,000 \t\t\t\t\...","16,081,000 \t\t\t\t\t\t...","136,817,000 \t\t\t\t\t\...","326,000,000 \t\t\t\t\t\...","99,720 \t\t\t\t\t\t\r\n...","2,413 \t\t\t\t\t\t\r\n\...",237 \t\t\t\t\t\t\r\n\t\...,"1,600 \t\t\t\t\t\t\r\n\..."


### 2. Standardize Column Names

In [3]:
# Convert to lowercase, strip whitespace, and replace spaces and hyphens with underscores
df.columns = df.columns.str.lower().str.strip().str.replace(r'\s+', '_', regex=True).str.replace('-', '_')
print(df.columns.tolist())

['country', 'power_index', 'total_population', 'total_military_manpower', 'fit_for_service', 'population_reaching_military_age_annually', 'active_personnel', 'reserve_personnel', 'paramilitary', 'total_military_aircraft', 'fighter_aircraft', 'attack_aircraft', 'transport_aircraft', 'trainer_aircraft', 'special_mission_aircraft', 'tanker_aircraft', 'total_military_helicopters', 'attack_helicopters', 'tanks', 'armored_fighting_vehicles', 'self_propelled_artillery', 'towed_artillery', 'rocket_projectors', 'total_naval_fleet', 'total_naval_fleet_tonnage_mt', 'aircraft_carriers', 'helicopter_carriers', 'submarines', 'destroyers', 'frigates', 'corvettes', 'coastal_patrol_craft', 'mine_warfare_craft', 'defense_budget_usd', 'external_debt_usd', 'purchasing_power_parity_usd', 'foreign_exchange_and_gold_reserves_usd', 'total_serviceable_airports', 'labour_force', 'major_ports_and_terminals', 'total_merchant_marine_fleet', 'railway_coverage_km', 'roadway_coverage_km', 'oil_production_bbl', 'oil_c

### 3. Clean Text and Convert Metrics to Numeric Formats

In [4]:
text_cols = ['country', 'continent', 'region', 'alliance']

def clean_currency_and_numbers(val):
    if pd.isna(val):
        return val
    if isinstance(val, str):
        # Remove non-numeric characters except period
        val = re.sub(r'[^\d.]', '', val)
        if val == '':
            return np.nan
        try:
            return float(val)
        except:
            return np.nan
    return val

for col in df.columns:
    if col not in text_cols:
        df[col] = df[col].apply(clean_currency_and_numbers)

### 4. Handle Missing/Null Values

In [5]:
# Fill missing metrics with 0
for col in df.columns:
    if col not in text_cols:
        df[col] = df[col].fillna(0)
    else:
        df[col] = df[col].fillna('Unknown')

missing_pct = df.isnull().sum().max()
print(f"Max missing values in any column after cleaning: {missing_pct}%")

Max missing values in any column after cleaning: 0%


### 5. Save Cleaned Dataset

In [6]:
output_path = 'military_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"Cleaned dataset successfully saved to {output_path}")

Cleaned dataset successfully saved to military_cleaned.csv
